<a href="https://colab.research.google.com/github/Abhiroop17/Deep-Learning-using-Python/blob/main/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
movies_data = pd.read_csv('/content/movies.csv')
ratings_data = pd.read_csv('/content/ratings.csv')

# Merge datasets on a common key (e.g., item ID)
data = pd.merge(movies_data, ratings_data, on='movieId')

# Split the data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)


In [8]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357274 sha256=7dabf24d9a746aa33d4a226425f3024a63991e25fa06fe159275a880cc305bae
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [11]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

# Convert the data into the Surprise dataset format
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(train_data[['userId', 'movieId', 'rating']], reader)

# Apply SVD algorithm
algo = SVD()
cross_validate(algo, surprise_data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

# Train on the entire dataset
trainset = surprise_data.build_full_trainset()
algo.fit(trainset)


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8794  0.8873  0.8732  0.8767  0.8844  0.8802  0.0051  
MAE (testset)     0.6773  0.6826  0.6734  0.6736  0.6799  0.6774  0.0035  
Fit time          1.04    1.08    1.03    1.22    1.65    1.20    0.23    
Test time         0.13    0.19    0.09    0.13    0.14    0.14    0.03    


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Apply TF-IDF to item attributes
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_data['genres'])

# Calculate similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Function to get recommendations
def get_recommendations(user_id, num_recommendations=10):
    user_items = train_data[train_data['userId'] == user_id]['movieId']
    sim_scores = cosine_sim[user_items].mean(axis=0)
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    recommended_items = [i[0] for i in sim_scores[:num_recommendations]]
    return item_data.iloc[recommended_items]

# Example recommendation for a user
recommendations = get_recommendations(user_id=1)
print(recommendations)


      userId  movieId  rating   timestamp
433        4     2282     1.0   945629238
4009      24   134393     2.5  1458941460
41         1      736     3.0   964982653
278        3     2090     0.5  1306464261
3852      23     2686     4.0  1107164093
4247      28      635     3.0  1242288183
6082      42     1892     3.0   996220343
7751      51     8533     5.0  1230930740
8284      57     1610     4.0   965796793
8575      58      356     4.0   847718461


In [21]:
def hybrid_recommendation(userId, movieId, num_recommendations=10):
    # Combine collaborative filtering and content-based filtering results
    cf_recommendations = algo.predict(userId, movieId)  # Example item_id should be replaced with actual item IDs
    cb_recommendations = get_recommendations(userId, num_recommendations)

    # Weighted combination of recommendations
    # Convert cb_recommendations to a DataFrame with a 'score' column
    cb_df = cb_recommendations.copy()
    cb_df['score'] = cb_df['rating'].astype(float)  # Assuming 'rating' column exists in cb_recommendations

    # Extract the score from the cf_recommendations
    cf_score = cf_recommendations.est

    # Combine scores (simple averaging for demonstration)
    cb_df['score'] = (cb_df['score'] + cf_score) / 2
    final_recommendations = cb_df.sort_values(by='score', ascending=False)

    return final_recommendations.head(num_recommendations)

# Example hybrid recommendation for a user (provide a valid movieId)
hybrid_recommendations = hybrid_recommendation(userId=1, movieId=110)  # Replace 110 with an actual movieId
print(hybrid_recommendations)

      userId  movieId  rating   timestamp     score
7751      51     8533     5.0  1230930740  4.769877
3852      23     2686     4.0  1107164093  4.269877
8284      57     1610     4.0   965796793  4.269877
8575      58      356     4.0   847718461  4.269877
41         1      736     3.0   964982653  3.769877
4247      28      635     3.0  1242288183  3.769877
6082      42     1892     3.0   996220343  3.769877
4009      24   134393     2.5  1458941460  3.519877
433        4     2282     1.0   945629238  2.769877
278        3     2090     0.5  1306464261  2.519877


In [23]:
# Function to retrain the model
def retrain_model(new_data):
    global algo
    # Merge new data and retrain the collaborative filtering model
    updated_data = pd.concat([train_data, new_data])
    surprise_data = Dataset.load_from_df(updated_data[['userId', 'movieId', 'rating']], reader)
    trainset = surprise_data.build_full_trainset()
    algo.fit(trainset)

# Example of adding new data and retraining
new_data = pd.read_csv('/content/movies.csv')
retrain_model(new_data)

In [26]:
# Assuming 'timestamp' is the extra column in test_data
testset = surprise_data.construct_testset(test_data[['userId', 'movieId', 'rating', 'timestamp']].values)

predictions = algo.test(testset)

# Calculate RMSE
rmse = accuracy.rmse(predictions)
print(f'Root Mean Square Error: {rmse}')

RMSE: 1.8243
Root Mean Square Error: 1.824275869335965
